![Practicum AI Logo image](https://github.com/PracticumAI/practicumai.github.io/blob/main/images/logo/PracticumAI_logo_250x50.png?raw=true) <img src="https://github.com/PracticumAI/practicumai.github.io/blob/84b04be083ca02e5c7e92850f9afd391fc48ae2a/images/icons/practicumai_computer_vision.png?raw=true" alt="Practicum AI: Computer Vision icon" align="right" width=50>
***

# Transfer Learning Concepts

You may recall *Practicum AI*"s heroine Amelia, the AI-savvy nutritionist. At the end of our *[Deep Learning Foundations course](https://practicumai.org/courses/deep_learning/)*, Amelia was helping with a computer vision project. If only she had known about transfer learning, it could have saved her a lot of time! In this notebook, we will get some hands-on experience with transfer learning and show you how to use it to improve your workflows.

![Figure 2 of the AgriNet paper used as the cover image for this notebook. Figure 2 depicts using transfer learning to make a computer vision model more efficient](images/agrinet_figure-cover.jpg)


## AI Pathway review for Transfer Learning & AgriNet 

If you have taken our [Getting Started with AI course](https://practicumai.org/courses/getting_started/), you may remember this figure of the AI Application Development Pathway. Let's take a quick review of how we will apply this to our case study of AgriNet and it's use of transfer learning.

![AI Application Development Pathway image showing the 7 steps in developing an AI application](https://practicumai.org/getting_started/images/application_dev_pathway.png)

1. **Choose a problem to solve:** In this example, we will be trying to make a computer vision model that can recognize images of plants, and categorize them as "healthy", "diseased", and a few other class categories that are plant-specific. 
2. **Gather data:** The data for the example comes from [HuggingFace](https://www.huggingface.co//), a great repository of datasets, code, and models.
3. **Clean and prepare the data:** In the *Deep Learning Foundations* course, we assumed that this was done for us. One issue that we ran into was that of class imbalance. Here, the (probably very tired) researchers that created the AgriNet dataset have already balanced the classes for us!
4. **Choose a model:** In the *Deep Learning Foundations* course, we presented the model with little detail. Here, we will use a Convolutional Neural Network (CNN) as our model, trained from scratch as a baseline, and compare it to a pre-trained VGG19 model that we've fine-tuned on the AgriNet dataset. VGG19 (Visual Geometry Group, with 19 layers) is a specific computer vision model architecture, and is pictured in the Cover Image above!
   * In the step where you'd choose a model, one can approach this in two ways:
      * **Train from scratch:** This is where you start with a randomly initialized model and train it on your data. This can be computationally expensive and time-consuming.
      * **Transfer learning:** This is where you start with a pre-trained model and fine-tune it on your data. This is often faster and requires less data.
5. **Train the model:** As mentioned in step 4, we'll demonstrate two approaches in this notebook:
      - Training a baseline model from scratch.
      - Fine-tuning a VGG19 model pre-trained on ImageNet, a domain-specific dataset.
6. **Evaluate the model:** We will use the metrics we gather to make decisions about the model. 
7. **Deploy the model:** We won't get to this stage in this course, but ideally we would end up with a model that could be deployed and achieve relatively good accuracy at solving crop classification problems.


### A Refresher

If you need a refresher, or haven't taken the *Deep Learning Foundations* course, the final notebook is part of this repository: [DLF_01.1_bees_vs_wasps.ipynb](DLF_01.1_bees_vs_wasps.ipynb).

### A Quick Primer on the Baseline Model
We've trained a simple convolutional neural network (CNN) from scratch as a baseline for comparison. If you have time and want to see how the CNN is set up, check out the [00.5_transfer_learning_helper.ipynb](00.5_transfer_learning_helper.ipynb) notebook that is included in this repository. 

Strictly speaking, a thorough knowledge of CNNs is not required for this notebook, but if you're interested in learning more, we recommend the our [PracticumAI: Computer Vision](https://github.com/PracticumAI/computer_vision) Intermediate course. That said, with *any* machine learning work, the better you understand the model, the better you can tune it to your needs.


### 1. Import the libraries we will use

As always, we will start by importing the libraries we will use in this notebook.

In [1]:
import numpy as np
import time
import copy
import tqdm
import os

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.models import VGG19_Weights
from torchvision import datasets, transforms, models
from PIL import ImageFile

import requests
import zipfile

import matplotlib.pyplot as plt

# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")

c:\Users\i.lutticken\AppData\Local\anaconda3\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Using CPU


### 2. Getting the data

Gotta have data to train, validate and test our models! We will use the AgriNet dataset, which is a dataset of images of plants that are categorized as "healthy", "diseased", and a few other categories.

In [2]:
# Download the dataset, extract it to the data folder and remove the zip file
download_path = "https://data.rc.ufl.edu/pub/practicum-ai/Transfer_Learning_Intermediate/agrinet_curated.zip"
zip_path = "data/agrinet_curated.zip"
data_path = "data"

# Paths to dataset
train_dir = os.path.join(data_path, "agri_net_train")
val_dir = os.path.join(data_path, "agri_net_val")
test_dir = os.path.join(data_path, "agri_net_test")

# Check if the data is already loaded
if not (
    os.path.exists(train_dir) and os.path.exists(val_dir) and os.path.exists(test_dir)
):
    # Create the data directory if it does not exist
    if not os.path.exists(data_path):
        os.makedirs(data_path)

    # Download the zip file
    r = requests.get(download_path)
    with open(zip_path, "wb") as f:
        f.write(r.content)

    # Extract the zip file
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(data_path)

    # Remove the zip file
    os.remove(zip_path)
else:
    print("Data is already loaded.")

Data is already loaded.


### 3. Looking at the Data

We will take a look at the data to see what we are working with. This is a good practice to get a sense of the data and to see if there are any issues that need to be addressed.

In [3]:
def show_test_images(dataset=test_dir, num_rows=3, num_cols=3):
    """
    Display a grid of labeled images from the test set

    Args:
        dataset: Whichever dataset you'd like to sample,
                    defaults to the test dataset
        num_rows: Number of rows in the grid (default: 3)
        num_cols: Number of columns in the grid (default: 3)
    """

    # Create figure and axes
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 12))

    # Sample random images from the test dataset
    indices = np.random.choice(len(dataset), num_rows * num_cols, replace=False)

    # Iterate over the grid positions
    for i, idx in enumerate(indices):
        # Get the image and label
        img, label = dataset[idx]

        # Calculate row and column position
        row, col = i // num_cols, i % num_cols
        ax = axes[row, col]

        # Convert tensor to numpy and transpose from (C,H,W) to (H,W,C)
        img = img.numpy().transpose((1, 2, 0))

        # Denormalize the image
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)

        # Display the image
        ax.imshow(img)
        ax.set_title(class_names[label])
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# Example usage
show_test_images(test_dir)

ValueError: not enough values to unpack (expected 2, got 1)

: 

### 4. Create the Data Loaders

Next, we'll create the data loaders for the training, validation and test sets. We'll use the `DataLoader` class from `torch.utils.data` to create the data loaders, and the `transforms` module from `torchvision` to apply transformations to the images.

In [ ]:
# Define PyTorch data transforms
data_transforms = {
    "train": transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    ),
    "val": transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    ),
    "test": transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    ),
}

# Load PyTorch datasets
image_datasets = {
    "train": datasets.ImageFolder(train_dir, data_transforms["train"]),
    "val": datasets.ImageFolder(val_dir, data_transforms["val"]),
    "test": datasets.ImageFolder(test_dir, data_transforms["test"]),
}

# Create PyTorch data loaders
dataloaders = {
    "train": torch.utils.data.DataLoader(
        image_datasets["train"],
        batch_size=128,
        shuffle=True,
        pin_memory=True,
        num_workers=2,
    ),
    "val": torch.utils.data.DataLoader(
        image_datasets["val"],
        batch_size=128,
        shuffle=False,
        pin_memory=True,
        num_workers=2,
    ),
    "test": torch.utils.data.DataLoader(
        image_datasets["test"],
        batch_size=128,
        shuffle=False,
        pin_memory=True,
        num_workers=2,
    ),
}

### Transfer Learning with AgriNet

We'll use the VGG19 model pre-trained on the AgriNet dataset, which is domain-specific to agriculture. Domain-specific pre-training captures features relevant to agricultural tasks, such as plant patterns and disease characteristics, which can further improve model performance compared to generic pre-trained models. Similar to the ImageNet approach, we'll freeze the base layers of the AgriNet model, add custom classification layers, and fine-tune the model on our dataset.

## Baseline Model

We'll train a simple convolutional neural network from scratch and use it as our baseline for performance comparison.

### Performance Comparison

We'll compare the performance of the three models (baseline, ImageNet pre-trained, and AgriNet pre-trained) using metrics like accuracy and F1-score. This step helps quantify the benefits of transfer learning and highlights the impact of using domain-specific pre-trained models. We'll evaluate each model on the test set and visualize the results using performance metrics and charts.

In [ ]:
# Handle truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True


# Define baseline model using PyTorch
class BaselineModel(nn.Module):
    def __init__(self, num_classes):
        super(BaselineModel, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 112 * 112, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


num_classes = len(image_datasets["train"].classes)
baseline_model_pt = BaselineModel(num_classes).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseline_model_pt.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# Initialize TensorBoard writer
writer = SummaryWriter()

# Define early stopping parameters
early_stopping_patience = 3
best_loss = float("inf")
patience_counter = 0

In [ ]:
# Train the baseline model using PyTorch
num_epochs = 5  # Reduced for demonstration
for epoch in range(num_epochs):
    baseline_model_pt.train()
    running_loss = 0.0
    progress_bar = tqdm.tqdm(
        dataloaders["train"], desc=f"Epoch {epoch+1}/{num_epochs}", leave=False
    )
    for inputs, labels in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = baseline_model_pt(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        progress_bar.set_postfix(loss=loss.item())
    scheduler.step()
    epoch_loss = running_loss / len(image_datasets["train"])
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

    # Log the loss to TensorBoard
    writer.add_scalar("Loss/train", epoch_loss, epoch)

    # Early stopping check
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= early_stopping_patience:
            print("Early stopping triggered")
            break

# Close the TensorBoard writer
writer.close()

### Conclusion: Key Insights

- Transfer learning significantly improves performance compared to training from scratch, especially with limited data.
- Domain-specific pre-training (e.g., AgriNet) can further enhance accuracy and generalization for specialized tasks.
- These findings demonstrate the importance of transfer learning in tackling real-world challenges in agriculture.

## Transfer Learning with ImageNet

We'll use a pre-trained VGG19 model with ImageNet weights and fine-tune it on our dataset.

In [ ]:
print(imagenet_model_pt)

In [ ]:
# Load pre-trained VGG19 model using PyTorch
weights = VGG19_Weights.IMAGENET1K_V1
imagenet_model_pt = models.vgg19(weights=weights).to(device)

# Freeze base layers
for param in imagenet_model_pt.parameters():
    param.requires_grad = False

# Add custom top layers
num_ftrs = imagenet_model_pt.classifier[0].in_features
imagenet_model_pt.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 128),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(128, num_classes),
).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(imagenet_model_pt.classifier.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# Train the model using PyTorch
num_epochs = 5  # Reduced for demonstration
for epoch in range(num_epochs):
    imagenet_model_pt.train()
    running_loss = 0.0
    progress_bar = tqdm.tqdm(
        dataloaders["train"], desc=f"Epoch {epoch+1}/{num_epochs}", leave=False
    )
    for inputs, labels in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = imagenet_model_pt(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        progress_bar.set_postfix(loss=loss.item())
    scheduler.step()
    epoch_loss = running_loss / len(image_datasets["train"])
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

## Conclusion

In this notebook, we demonstrated the benefits of transfer learning in agricultural tasks. The AgriNet pre-trained model outperformed the ImageNet model and the baseline, showing the importance of domain-specific pre-training for specialized applications.